# Get full human proteome and ioforms from UniProt

In this Notebook we will use fasta files downloaded from the UniProt website to create lists of canonical proteins and protein isoforms. All the data was downloaded from https://www.uniprot.org on the 31.06.2026 and 05.04.2026. The proteins that are included in the database are limited to human proteins that are UniProtKB reviewed (Swiss-Prot). There are 15 proteins that are not included in the human proteome by UniProt because they do not map to the reference genome (Query: organism_id:9606 AND reviewed:true NOT proteome:up000005640). We have also decided to not include these proteins in our database.

# Setup

In [ ]:
import sys
import os
import session_info

# Add the '0_functions' folder to sys.path
sys.path.append(os.path.join(os.getcwd(), '..', '00_functions'))

In [ ]:
import pandas as pd
from functions import read_fastafile

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/01_UniProt"):
    print("WARNING: The working directory is not set to the '01_UniProt'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '01_Uniprot' folder (\"{cwd}\").")

In [ ]:
# Data directory for fasta data
raw_data_dir = "data/raw_data"
duplicates_dir = "data/sequence_duplicates"
fasta_dir = "data/raw_data/fasta"

In [ ]:
# Get query dataframes
full_proteome_query = pd.read_csv(os.path.join(raw_data_dir,'human_proteome_list.csv'))
full_proteome_isoforms_query = pd.read_csv(os.path.join(raw_data_dir, 'human_proteome_isoforms_list.csv'))

## Read in Fasta files

In [ ]:
fasta_canonical = read_fastafile(os.path.join(fasta_dir, 'uniprotkb_canonical_2026_03_31.fasta'))
fasta_full_proteome = read_fastafile(os.path.join(fasta_dir, 'uniprotkb_full_proteome_2026_03_31.fasta'))
fasta_alternative_product = read_fastafile(os.path.join(fasta_dir, 'uniprotkb_alternative_products_isoforms_2026_03_31.fasta'))
fasta_alternative_splicing = read_fastafile(os.path.join(fasta_dir, 'uniprotkb_alternative_splicing_isoforms_2026_03_31.fasta'))

We will take a quick look at the different dataframes created from the fasta files. The number of entries is especially of interest.

In [ ]:
fasta_canonical

In [ ]:
fasta_full_proteome

In [ ]:
fasta_alternative_product

In [ ]:
fasta_alternative_splicing

In [ ]:
#combine alternative_product and alternative_splicing df into one isoform df
fasta_isoforms = pd.concat([fasta_alternative_product, fasta_alternative_splicing], ignore_index=True)

# Drop duplicates based on Sequence
# keep='first' means if a sequence is in both, we keep the one from the first file loaded
fasta_isoforms_unique = fasta_isoforms.drop_duplicates(subset=["seq"], keep="first").reset_index(drop=True)

print(f"Total sequences after deduplication: {len(fasta_isoforms_unique)}")

## Check if sizes of dataframes from query and fasta match (canonical and isoform) 

In [ ]:
#Full proteome
if len(fasta_full_proteome["ID"].unique()) == len(full_proteome_query["ID"].unique()):
    print("Lengths match!")
else:
    print(f"Mismatch: {len(fasta_full_proteome['ID'].unique())} vs {len(full_proteome_query['ID'].unique())}")

In [ ]:
# Canonical
if len(fasta_canonical["ID"]) == len(full_proteome_query["ID"].unique()):
    print("Lengths match!")
else:
    print(f"Mismatch: {len(fasta_canonical['ID'])} vs {len(full_proteome_query['ID'].unique())}")

In [ ]:
# Isoforms (indivisual fasta)
if len(fasta_alternative_product["ID"]+fasta_alternative_splicing["ID"]) == len(full_proteome_isoforms["isoform_id"].unique()):
    print("Lengths match!")
else:
    print(f"Mismatch: {len(fasta_alternative_product['ID']+fasta_alternative_splicing['ID'])} vs {len(full_proteome_isoforms['isoform_id'].unique())}")

In [ ]:
# Isoforms (merged fasta)
if len(fasta_isoforms_unique["ID"]) == len(full_proteome_isoforms["isoform_id"].unique()):
    print("Lengths match!")
else:
    print(f"Mismatch: {len(fasta_isoforms_unique['ID'])} vs {len(full_proteome_isoforms['isoform_id'].unique())}")

## Look into mismatch in full proteome
Number of IDs found from fasta file found in isoform query df does not match number of sequences from fasta file in isoform query file. This is the case because there is a mistake in the original query. After looking into this we have decided to use the downloaded fasta file and not work with the queried data.

In [ ]:
#Full proteome unique sequences
if len(fasta_full_proteome["seq"].unique()) == len(full_proteome["Sequence"].unique()):
    print("Lengths match!")
    print(len(fasta_full_proteome["seq"].unique()))
else:
    print(f"Mismatch: {len(fasta_full_proteome['seq'].unique())} vs {len(full_proteome['Sequence'].unique())}")

In [ ]:
# Duplicates in the FASTA dataframe
fasta_duplicates = fasta_full_proteome[fasta_full_proteome.duplicated(subset=['seq'], keep=False)]

# Duplicates in the Proteome dataframe
query_duplicates = full_proteome[full_proteome.duplicated(subset=['Sequence'], keep=False)]

print(f"Fasta duplicate rows: {len(fasta_duplicates)}")
print(f"Query duplicate rows: {len(query_duplicates)}")

In [ ]:
fasta_duplicates.to_csv(os.path.join(duplicates_dir, 'full_proteome_duplicates.csv'), index=False)

These 139 proteins were all checked manually in uniprot and will be further processed in notebook 1.3 where I created dataframes that have unique IDs and unique sequences. 

In [ ]:
duplicates_list = pd.read_csv(os.path.join(duplicates_dir, 'duplicates_list_01042026.csv'))

## Check if IDs and sequences match between the fasta canonical and fasta isoform dataframes

In [ ]:
#check if all the canonical protein IDs are in the full proteome df
fasta_canonical["ID"].isin(fasta_full_proteome["ID"]).all()

In [ ]:
#check how many of the canonical protein IDs are in the isoform df
print(f"{fasta_canonical['ID'].isin(fasta_isoforms['ID']).sum()} out of {len(fasta_canonical['ID'])} canonical IDs found in isoform df.")

This is what we would expect -> matches number of canonical IDs that have alternative products (isoforms). Since this is the number of canonical IDs for alternative products not alternative slicing, let's check how many canonical ids are found in these dataframes individually.

In [ ]:
#check how many of the canonical protein IDs are in the alternative products df
print(f"{fasta_canonical['ID'].isin(fasta_alternative_product['ID']).sum()} out of {len(fasta_canonical['ID'])} canonical IDs found in alternative products df.")

#check how many of the canonical protein IDs are in the alternative splicing df
print(f"{fasta_canonical['ID'].isin(fasta_alternative_splicing['ID']).sum()} out of {len(fasta_canonical['ID'])} canonical IDs found in alternative splicing df.")

#check how many of the canonical protein sequences are in the alternative splicing df
print(f"{fasta_canonical['seq'].isin(fasta_alternative_splicing['seq']).sum()} out of {len(fasta_canonical['seq'])} canonical sequences found in alternative splicing df.")
print(f"{fasta_alternative_splicing['seq'].isin(fasta_canonical['seq']).sum()} out of {len(fasta_canonical['seq'])} alternative splicing sequences found in canonical df.")

In [ ]:
#check how much overlap in IDs there is between the alternative splicing and the alternative product dfs
print(f"{fasta_alternative_splicing['ID'].isin(fasta_alternative_product['ID']).sum()} alternative splicing IDs found in alternative products df.")
print(f"{fasta_alternative_product['ID'].isin(fasta_alternative_splicing['ID']).sum()} alternative products IDs found in alternative splicing df.")

#check how much overlap in seqeunces there is between the alternative splicing and the alternative product dfs
print(f"{fasta_alternative_splicing['seq'].isin(fasta_alternative_product['seq']).sum()} alternative splicing sequences found in alternative products df.")
print(f"{fasta_alternative_product['seq'].isin(fasta_alternative_splicing['seq']).sum()} alternative products sequences found in alternative splicing df.")



This means that the sequences in the alternative splicing df fully match the sequences in the alternative products df so I will not use the alternative splicing df further. 

The 10588 alternative products sequences found in alternative splicing df is likely because of a sequence duplicate. We'll look into this in the next part.

## Investigating the alternative products df and the full proteome df further

In [ ]:
fasta_alternative_product['ID'].isin(fasta_full_proteome['ID']).all()

In [ ]:
#Check for duplicates in alternative products (isoform) df
isoform_duplicates = fasta_alternative_product[fasta_alternative_product.duplicated(subset=['seq'], keep=False)]

print(f"isoform duplicate rows: {len(isoform_duplicates)}")

In [ ]:
isoform_duplicates

In [ ]:
isoform_duplicates.to_csv(os.path.join(duplicates_dir, 'isoform_duplicates.csv'), index=False)

Since the Q7Z3S9/P0DPK4-2 duplicate contains a canonical sequence, we already processed it for the unique canonical, sequences. The same goes for the P62684/Q9YNA8 duplicate. We will create a unique ID for the A6NHP3-2/Q495Y8-2 duplicate, which will later map to both canonical IDs. 